# An Elementary Introduction to `CVXPY`

## Detour: Symbolic Convexity Checking Using `SymPy`
$\newcommand{\ds}{\displaystyle}$
For differentiable function $f$, check if its Hessian is positive-semidefinite: $\;\ds\nabla^2 f\succcurlyeq 0$
- $\ds f(x, y) = x \log\frac{x}{y},\quad x > 0,\quad y > 0$

In [ ]:
from sympy.abc import w, x, y, z  
from sympy import ordered, hessian, log, exp
from sympy import init_printing
init_printing(use_latex='mathjax')

expr = x * log(x / y)
v = list(ordered(expr.free_symbols))
display(hessian(expr, v))
hessian(expr, v).eigenvals()

### Your Turn: Check the Convexity of the Following Functions
- $\ds f(x) = x\log x$, $\quad x > 0$

In [ ]:
expr = x * log(x)
v = list(ordered(expr.free_symbols))
display(hessian(expr, v))
hessian(expr, v).eigenvals()

- $\ds f(x, y) = \frac{x^2}{y}$, $\quad y > 0$

In [ ]:
expr = x**2 / y
v = list(ordered(expr.free_symbols))
display(hessian(expr, v))
hessian(expr, v).eigenvals()

- $\ds f(x, y, z) = \log\,(e^{x} + e^{y} + e^{z})$

In [ ]:
expr = log(exp(x) + exp(y) + exp(z))
v = list(ordered(expr.free_symbols))
display(hessian(expr, v))
hessian(expr, v).eigenvals()

## First Encounter

`CVXPY` [official site](https://www.cvxpy.org/index.html) 

In [ ]:
import cvxpy as cp

# Create two scalar optimization variables.
x = cp.Variable()
y = cp.Variable()

# Create two constraints.
constraints = [x + y == 1,
               x - y >= 1]

# Form objective.
obj = cp.Minimize((x - y)**2)

# Form and solve problem.
prob = cp.Problem(obj, constraints)
prob.solve()  
# Return optimal values
print("status:", prob.status, "\noptimal value:", prob.value, "\noptimal variables:", x.value, y.value)

### In Practice

In [ ]:
import cvxpy as cp

# Create the optimization variable x of length 2.
x = cp.Variable(2)
prob = cp.Problem(cp.Minimize((x[0] - x[1])**2), [x[0] + x[1] == 1, x[0] - x[1] >= 1])
prob.solve()  
# Return optimal values
print("status:", prob.status, "\noptimal value:", prob.value, "\noptimal variables:", x.value)

## Showcases

### 01. Linear Programming (LP)

Standard form
$$\begin{array}{ll}
    \mbox{minimize}   & c^\top x \\
    \mbox{subject to} & Ax \preccurlyeq b.
    \end{array}$$
where $A \in \mathbb{R}^{m \times n}$, $b \in \mathbb{R}^m$, and $c \in \mathbb{R}^n$; $x \in \mathbb{R}^{n}$ is the optimization variable.

In [ ]:
# Import packages.
import cvxpy as cp
import numpy as np

# Generate a random non-trivial linear program.
m = 15
n = 10
np.random.seed(1)
s0 = np.random.randn(m)
lamb0 = np.maximum(-s0, 0)
s0 = np.maximum(s0, 0)
x0 = np.random.randn(n)
A = np.random.randn(m, n)
b = A@x0 + s0
c = -A.T@lamb0

# Define and solve the CVXPY problem.
x = cp.Variable(n)
prob = cp.Problem(cp.Minimize(c.T@x), [A@x <= b])
prob.solve()

# Print result.
print("\nThe optimal value is", prob.value, "\nA solution x is", x.value)

### Your Turn: LP Formulations 
\begin{align*}
  \text{maximize}\quad& 5 x_1 + 4 x_2 + 3 x_3 \\
  \text{subject to}\quad& 2 x_1 + 3 x_2 + x_3 \leqslant 5 \\
  \qquad\qquad\quad& 4 x_1 + x_2 + 2 x_3 \leqslant 11 \\
  \qquad\qquad\quad& 3 x_1 + 4 x_2 + 2 x_3 \leqslant 8 \\
  \qquad\qquad\quad& x_1\geqslant 0,\quad x_2\geqslant 0,\quad x_3\geqslant 0
\end{align*}
- The maximum: $13$. The optimizer: $(2,\,0,\,1)$. Using two ways (Index & Matrix) to do this.
- You may need `x = cp.Variable(3, nonneg=True)` and `np.round()`.


In [ ]:
import numpy as np
import cvxpy as cp

x = cp.Variable(3, nonneg=True)

A = np.array([[2, 3, 1], [4, 1, 2], [3, 4, 2]])
c = np.array([5, 4, 3])
b = np.array([5, 11, 8])

# TODO: Index & Matrix
prob = cp.Problem(cp.Maximize(c@x), [A@x <= b])
#

prob.solve()
print(np.round(prob.value))
print(np.round(x.value))

#### 01.1 0-1 Knapsack Problem

- A set of $n$ items number from $1$ up to $n$, each with a weight $w_i$ and a value $v_i$. The maximum weight capacity of the knapsack is $W$ and the number $x_i$ of each kind of item inside is zero or one.
- The mathematical formulation of this 0-1 knapsack problem is:
  \begin{align*}
    \text{maximize}\quad&\sum_i v_i x_i\\
    \text{subject to}\quad&\sum_i w_i x_i\leqslant W\;\text{ and }\;x_i\in\{0, 1\},\;i=1,\,2,\,\ldots,\,n
  \end{align*}
- An example (Martello-Toth Example 2.1):
  \begin{align*}
   w &= [2,\,20,\,20,\,30,\,40,\,30,\,60,\,10] \\
   v &= [15,\,100,\,90,\,60,\,40,\,15,\,10,\,1] \\
   W &= 102
  \end{align*}
  The maximum: $280$, the optimal solution $\ds x = [1,\,1,\,1,\,1,\,0,\,1,\,0,\,0]$.

In [ ]:
import cvxpy as cp
v = [15, 100, 90, 60, 40, 15, 10, 1]
w = [2, 20, 20, 30, 40, 30, 60, 10]
W = 102
x = cp.Variable(len(w), boolean=True)
prob = cp.Problem(cp.Maximize(cp.sum(v @ x)), [cp.sum(w @ x) <= W])
prob.solve()
print(prob.value)
print(x.value)

### 02. Least-Squares / Linear Regression

In a least-squares / linear regression problem, we have measurements $A \in \mathbb{R}^{m \times n}$ and $b \in \mathbb{R}^m$ and seek a vector $x \in \mathbb{R}^{n}$ such that $Ax$ is close to $b$. Closeness is defined as the sum of the squared differences:
$$ \sum_{i=1}^m (a_i^\top x - b_i)^2, $$
also known as the $\ell_2$-norm squared, $\|Ax - b\|_2^2$.

We find the optimal $x$ by solving the optimization problem
$$  
    \begin{array}{ll}
    \mbox{minimize}   & \|Ax - b\|_2^2.
    \end{array}
$$
Let $x^\star$ denote the optimal $x$. The quantity $r = Ax^\star - b$ is known as the residual. If $\|r\|_2 = 0$, we have a perfect fit.

In [ ]:
# Import packages.
import cvxpy as cp
import numpy as np

# Generate data.
m = 20
n = 15
np.random.seed(1)
A = np.random.randn(m, n)
b = np.random.randn(m)

# Define and solve the CVXPY problem.
x = cp.Variable(n)
cost = cp.sum_squares(A@x - b)
prob = cp.Problem(cp.Minimize(cost))
prob.solve()

# Print result.
print("\nThe optimal value is", prob.value, "\nThe optimal x is", x.value, 
      "\nThe norm of the residual is", cp.norm(A@x - b, p=2).value)

### Your Turn: Ridge & LASSO  

- We wish to recover a sparse vector $x \in \mathbb{R}^n$ from measurements $y\in\mathbb{R}^m$. Our measurement model tells us that 
$$y = Ax + v$$
where $A \in \mathbb{R}^{m \times n}$ is known and $v \in \mathbb{R}^m$ is unknown measurement error.
- The entries of $v$ are sampled from the normal distribution with mean $0$ and standard deviation $\sigma = 1$.
- We first try to recover $x$ by solving the Ridge problem
\begin{align*}
  \text{mimimize}\quad \|Ax-y\|^2_2 + \gamma\|x\|^2_2
\end{align*}
- A more effective approach is to solve the LASSO problem
\begin{align*}
  \text{mimimize}\quad \|Ax-y\|^2_2 + \gamma\|x\|_1
\end{align*}
- Estimate $x$ from $y$ using both Ridge and LASSO regression. Your task is to provide the definitions of `ridge_loss` and `lasso_loss`.
- We first let `gamma` equals $1$. Can you find a better `gamma`?
- Suggestion: [`CVXPY` functions lookup.](https://www.cvxpy.org/tutorial/functions/index.html)

In [ ]:
# Ridge regression vs. LASSO to estimate sparse x.
import numpy as np
import cvxpy as cp
import scipy.linalg as la
import scipy.sparse as sp

np.random.seed(1)
n = 4000
m = 2000
true_x = 100 * sp.rand(n, 1, 0.1).toarray().flatten()
A = np.random.randn(m, n)
v = np.random.normal(0, 1, m)
y = A @ true_x + v

x = cp.Variable(n)
gamma = 1

ridge_loss = None # FIXME
ridge = cp.Problem(cp.Minimize(ridge_loss))
ridge.solve()
x_ridge = x.value

lasso_loss = None # FIXME
lasso = cp.Problem(cp.Minimize(lasso_loss))
lasso.solve()
x_lasso = x.value

import matplotlib.pyplot as plt
%matplotlib inline
plt.semilogy(range(n), np.sort(np.abs(true_x - x_ridge)),  label="ridge errors")
plt.semilogy(range(n), np.sort(np.abs(true_x - x_lasso)),  label="lasso errors")
plt.legend()
plt.show()

### 03. Geometric Programming

#### 03.1 Maximizing the Volume of a Box

We maximize the shape of a box with height $h$, width $w$, and depth $d$, with limits on the wall area $2(h\cdot w + h\cdot d)$
and the floor area $w\cdot d$, subject to bounds on the aspect ratios $\ds\frac{h}{w}$ and $\ds\frac{w}{d}$. The optimization problem is

$$
\begin{array}{ll}
\mbox{maximize} & h\cdot w\cdot d \\
\mbox{subject to} & 2\,(h\cdot w + h\cdot d) \leqslant A_{\text{wall}}, \\
& w\cdot d \leqslant A_{\text{flr}}, \\
& \alpha \leqslant \frac{h}{w} \leqslant \beta, \\
& \gamma \leqslant \frac{d}{w} \leqslant \delta.
\end{array}
$$

In [ ]:
import cvxpy as cp

# Problem data
A_wall = 100
A_flr = 10
alpha = 0.5
beta = 2
gamma = 0.5
delta = 2

h = cp.Variable(pos=True)
w = cp.Variable(pos=True)
d = cp.Variable(pos=True)

constraints = [
    2 * (h * w + h * d) <= A_wall,
    w * d <= A_flr,
    h / w >= alpha,
    h / w <= beta,
    d / w >= gamma,
    d / w <= delta
]
problem = cp.Problem(cp.Maximize(h * w * d), constraints)
print(problem)
problem.solve()

#### Test if the problem is of "disciplined convex programming (dcp)" or "disciplined geometric programming (dgp)" & Solve the dgp problem

In [ ]:
print('Is the problem dcp? ', problem.is_dcp(), '\nIs the problem dgp? ', problem.is_dgp())

# Solve the dgp problem
problem.solve(gp=True)
print('problem.value: ', problem.value, 
      '\nh.value:', h.value, '\nw.value:', w.value, '\nd.value:', d.value)

#### 03.2 Perron-Frobenius Matrix Completion Problem

- Let matrix $X\in\mathbb{R}^{n\times n}$ have positive entries. The Perron-Frobenius theorem
states that $X$ has a positive real eigenvalue $\lambda_{\text{pf}}$ equal to its spectral radius, i.e., the magnitude of its largest eigenvalue. We have

\begin{align*}
  \lambda_{\text{pf}} = \min\,\{\,\lambda\;|\; X v\leqslant\lambda v\,\;\text{for some $v > 0$}\,\}
\end{align*}

- Now we are given some entries of an elementwise positive matrix $A$, and the goal is to choose the missing entries so as to minimize the Perron-Frobenius eigenvalue:
  \begin{align*}
  \text{minimize}\quad&\lambda_{\text{pf}}(X) \\
  \text{subject to}\quad&\prod_{\substack{(i, j)\not\in\Omega}} X_{ij} = 1 \\
  \qquad\qquad\quad &X_{ij} = A_{ij},\quad (i, j)\in\Omega                           
  \end{align*}
  where $\Omega$ denote the set of indices $(i, j)$ for which $A_{ij}$ is known. 

- Here $A$ is
\begin{align*}
  A = \begin{pmatrix}1.0 & ? & 1.9 \\ ? & 0.8 & ? \\ 3.2 & 5.9 & ? \end{pmatrix}            
\end{align*}
  - Find the best missing entries of $A$. 
  - Note this is a geometric programming problem: `prob.solv(gp=True)`

In [ ]:
import cvxpy as cp

known_value_indices = tuple(zip(*[[0, 0], [0, 2], [1, 1], [2, 0], [2, 1]]))
known_values = [1.0, 1.9, 0.8, 3.2, 5.9]
x = cp.Variable((3, 3), pos=True)
obj = cp.pf_eigenvalue(x)
constraints = [x[known_value_indices] == known_values, 
               x[0, 1] * x[1, 0] * x[1, 2] * x[2, 2] == 1.,]
prob = cp.Problem(cp.Minimize(obj), constraints)
prob.solve(gp=True)
print("Optimal value: ", prob.value, "\nx:", x.value)

### Your Turn: Water-Filling Problem in Information Theory

- Aim: allocating power to a set of $n$ channels to maximise the total channel capacity
- $x_i$, $i=1,2,\ldots, n$ represents the power allocated to the $i$-th channel
- $\log(\alpha_i + x_i)$ gives the capacity of the $i$-th channel; $\alpha_i$ is given
- Constraints: $\ds\sum_{i=1}^n x_i = 1$, $x_i\geqslant 0$
- An example
  - Given $\ds\alpha = [0.8,\,1.0,\,1.2],\;$ the maximum channel capacity: $\;0.863,\;$ power allocation: $\;[0.533,\,0.333,\,0.133]$ 

In [ ]:
import cvxpy as cp
import numpy as np
np.set_printoptions(precision=3)

#alpha = np.array([1.1, 0.7, 0.25, 0.42, 0.53, 0.31, 0.53, 0.6, 1.05, 0.27])
alpha = np.array([0.8, 1.0, 1.2])

# TODO

prob.solve()
print("Optimal value:", prob.value, "\nx:", x.value)